In [1]:
from src.data.data_loader import (
    load_training_params,
    get_Xtrain_data,
    get_Xtest_data,
    get_ohe_Ytrain,
    get_weighted_Ytrain,
    get_ohe_Ytest,
    get_weighted_Ytest,
)
from src.utils.model_exporter import (
    export_weighted_supervised,
    export_ohe_supervised,
)
from src.data.process_data import add_noise
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
training_output = load_training_params()

# Training data
X_train = get_Xtrain_data()
ohe_Y_train = get_ohe_Ytrain()
weighted_Y_train = get_weighted_Ytrain()

# Test data
X_test = get_Xtest_data()
ohe_Y_test = get_ohe_Ytest()
weighted_Y_test = get_weighted_Ytest()

In [3]:
# Supervised models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Weighted_clf = RandomForestClassifier(n_estimators=300, random_state=42)
OHE_clf = RandomForestClassifier(n_estimators=300, random_state=42)

In [4]:
# Training
Weighted_scores = cross_val_score(
    Weighted_clf, X_train, weighted_Y_train, cv=cv, scoring="accuracy"
)
OHE_scores = cross_val_score(OHE_clf, X_train, ohe_Y_train, cv=cv, scoring="accuracy")
print(
    f"Mean Accuracy (Weighted): {Weighted_scores.mean():.4f} ± {Weighted_scores.std():.4f}"
)
print(f"Mean Accuracy (OHE): {OHE_scores.mean():.4f} ± {OHE_scores.std():.4f}")

Weighted_clf.fit(X_train, weighted_Y_train)
OHE_clf.fit(X_train, ohe_Y_train)
y_pred_Weighted = Weighted_clf.predict(X_test)
y_pred_OHE = OHE_clf.predict(X_test)

Mean Accuracy (Weighted): 0.9989 ± 0.0002
Mean Accuracy (OHE): 0.9997 ± 0.0001


In [5]:
# Test models
print("\n=== Weighted Model Test Results ===")
print(f"Accuracy: {accuracy_score(weighted_Y_test, y_pred_Weighted):.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(weighted_Y_test, y_pred_Weighted))
print("\nClassification Report:")
print(classification_report(weighted_Y_test, y_pred_Weighted))

print("\n=== OHE Model Test Results ===")
print(f"Accuracy: {accuracy_score(ohe_Y_test, y_pred_OHE):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(ohe_Y_test, y_pred_OHE))
print("\nClassification report:")
print(classification_report(ohe_Y_test, y_pred_OHE))


=== Weighted Model Test Results ===
Accuracy: 0.9990

Confusion matrix:
[[4379    0    3]
 [   0 1572    0]
 [  10    0 6835]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4382
           1       1.00      1.00      1.00      1572
           2       1.00      1.00      1.00      6845

    accuracy                           1.00     12799
   macro avg       1.00      1.00      1.00     12799
weighted avg       1.00      1.00      1.00     12799


=== OHE Model Test Results ===
Accuracy: 0.9998

Confusion Matrix:
[[4916    1    1]
 [   0 6311    0]
 [   1    0 1569]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4918
           1       1.00      1.00      1.00      6311
           2       1.00      1.00      1.00      1570

    accuracy                           1.00     12799
   macro avg       1.00      1.00      1.00     

In [6]:
# Add noise to test generalization
X_test_noisy = add_noise(
    X_test,
    numeric_cols=training_output["numeric_cols"],
    categorical_cols=training_output["categorical_cols"],
)
pred_noisy = OHE_clf.predict(X_test_noisy)
pred_noisy = Weighted_clf.predict(X_test_noisy)

consistency_Weighted = (y_pred_Weighted == pred_noisy).mean()
consistency_OHE = (y_pred_OHE == pred_noisy).mean()
print(
    f"Consistency of predictions in the presence of noise (Weighted): {consistency_Weighted:.4f}"
)
print(
    f"Consistency of predictions in the presence of noise (OHE): {consistency_OHE:.4f}"
)

Consistency of predictions in the presence of noise (Weighted): 0.9966
Consistency of predictions in the presence of noise (OHE): 0.3431


In [7]:
# Export models
export_ohe_supervised(OHE_clf)
export_weighted_supervised(Weighted_clf)